# Basic UDTF Generation

This notebook demonstrates how to generate UDTF code from a CDF Data Model using pygen-spark.

## Prerequisites

- Spark cluster with active Spark session
- CDF credentials in `config.toml` file
- CDF Data Model with Views


## Step 1: Install Dependencies


In [ ]:
# Install required packages
# Note: Restart Python kernel after installation when prompted

%pip install cognite-pygen-spark cognite-sdk


## Step 2: Load CDF Client


In [ ]:
from cognite.pygen import load_cognite_client_from_toml
from cognite.client.data_classes.data_modeling.ids import DataModelId

# Load client from TOML file
# config.toml format:
# [cognite]
# project = "<cdf-project>"
# tenant_id = "<tenant-id>"
# cdf_cluster = "<cdf-cluster>"
# client_id = "<client-id>"
# client_secret = "<client-secret>"

client = load_cognite_client_from_toml("config.toml")
print("✓ CDF client loaded")


## Step 3: Generate UDTFs


In [ ]:
from cognite.pygen_spark import SparkUDTFGenerator
from pathlib import Path

# Define data model
data_model_id = DataModelId(
    space="sailboat",
    external_id="sailboat",
    version="1"
)

# Create generator
generator = SparkUDTFGenerator(
    client=client,
    output_dir=Path("./generated_udtfs"),
    data_model=data_model_id,
    top_level_package="cognite_udtfs",
)

# Generate UDTFs
result = generator.generate_udtfs()

print(f"\n✓ Generated {result.total_count} UDTF(s)")
print(f"  Output directory: {result.output_dir}")
print("\nGenerated files:")
for view_id, file_path in result.generated_files.items():
    print(f"  - {view_id}: {file_path}")


## Step 4: Verify Generated Files

You can inspect the generated UDTF files to see the code structure.


In [ ]:
# List generated files
from pathlib import Path

output_dir = Path("./generated_udtfs")
if output_dir.exists():
    print("Generated UDTF files:")
    for file_path in sorted(output_dir.rglob("*.py")):
        print(f"  - {file_path}")
        
    # Show first few lines of a generated file as example
    if result.generated_files:
        first_file = list(result.generated_files.values())[0]
        print(f"\nFirst 20 lines of {first_file.name}:")
        with open(first_file, 'r') as f:
            for i, line in enumerate(f):
                if i < 20:
                    print(line.rstrip())
                else:
                    break
else:
    print("Output directory not found")
